# Ordered Logistic Regression Adoption Predictors Exploration with `mlcroissant`
This notebook demonstrates step-by-step exploration of the FAIR^2 dataset using the `mlcroissant` library, referencing all identifiers (record sets, fields, columns) using their `@id` values as per the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant library if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata.to_json()
print("\nDataset Title:")
print(metadata['name'])

print("\nDataset Description:")
print(metadata['description'])

print("\nData Collection Summary:")
print(metadata.get('dataCollection'))

print("\nLicense:")
print(metadata.get('license'))

print("\nTemporal Coverage:")
print(metadata.get('temporalCoverage'))

print("\nKeywords:")
print(metadata.get('keywords'))

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

- The dataset schema may contain multiple record sets (`recordSet`), each with unique fields and columns defined by their `@id`s.
- Here, we enumerate all record sets and preview their structure.

In [ ]:
# List available record sets and their @id
record_sets = dataset.metadata.record_sets

print("Available Record Sets (Referenced by @id):\n")
for rs in record_sets:
    print(f"Record Set Name: {getattr(rs, 'name', 'Unnamed')}\n  @id: {rs['@id']}")
    fields = getattr(rs, 'fields', [])
    print("  Fields:")
    for field in fields:
        print(f"    - {field['@id']} ({getattr(field, 'name', field['@id'])})")
    columns = getattr(rs, 'columns', [])
    if columns:
        print("  Columns:")
        for column in columns:
            print(f"    - {column['@id']} ({getattr(column, 'name', column['@id'])})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s discovered above.

In [ ]:
# Extract data from all record sets
import warnings
warnings.filterwarnings('ignore')

record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# For demonstration, we try to load records from each record set
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for record set '@id': {rs_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Could not load record set '@id': {rs_id}, error: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

- We demonstrate filtering, normalization, and grouping using `@id` identifiers.
- Replace `<numeric_field_id>` and `<group_field>` with actual IDs from previous cell outputs.

In [ ]:
# Select one record set and a numeric field (by @id)
# For demonstration, we'll select the first loaded record set with at least one numeric column
import numpy as np

if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    df = dataframes[selected_record_set_id]

    # Try to auto-detect a numeric field
    numeric_field_id = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break
    
    if numeric_field_id:
        print(f"Using numeric field '@id': {numeric_field_id}")
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())
        
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())
        
        # Try to group by a categorical column
        group_field_id = None
        for col in df.columns:
            if df[col].dtype == object and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            print(f"Grouping by field '@id': {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields found in the selected record set.")
else:
    print("No record sets loaded to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We plot the distribution of the selected numeric field and visualize group averages if available.

In [ ]:
import matplotlib.pyplot as plt

if 'df' in locals() and numeric_field_id:
    plt.figure(figsize=(8, 4))
    plt.hist(df[numeric_field_id].dropna(), bins=20, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    if 'grouped_df' in locals():
        plt.figure(figsize=(8,4))
        plt.bar(grouped_df[group_field_id], grouped_df[numeric_field_id], color='salmon')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook has demonstrated loading and exploration of a FAIR^2 dataset package using the `mlcroissant` library, referencing all data entities by their `@id`s.

Key findings and next steps:
- Successfully loaded dataset metadata and explored available record sets, their fields, and columns.
- Extracted tabular data from record sets, filtered and normalized numeric fields using their `@id`s.
- Visualized field distributions and group statistics.

This workflow can be extended for more complex analyses, modeling, or cross-dataset comparisons using Croissant and `mlcroissant`.